# Diversité des modèles du benchmark

Évaluer les domaines couverts, les liens entre problèmes et la variété des matrices de transition et de récompense. Les effectifs du catalogue, la sélection proposée et les instances effectivement analysées sont présentés séparément. Aucun solveur ni calcul de valeur optimale n'est lancé.

Utiliser le noyau Conda `benchmark`, avec les dépendances `mdpforge[notebooks]` installées depuis ce dépôt. Pour actualiser l'étude après une modification du code, redémarrer le noyau puis exécuter toutes les cellules. Les modèles et presets sont redécouverts à chaque exécution ; les nouveaux modèles ne demandent aucune modification de la liste de chargement.

Par défaut, l'inventaire couvre les deux catalogues et tous leurs presets ; l'analyse numérique porte sur `tiny`, sans plafond de calcul. Elle construit réellement les modèles et peut donc prendre du temps. L'analyse complète et les comparaisons d'actions peuvent demander beaucoup de mémoire et de temps. Mettre `ANALYZE_MATRICES = False` pour consulter uniquement l'inventaire.

In [ ]:
import gc
import hashlib
import inspect
import json
import platform
import random
import subprocess
import warnings
from datetime import datetime, timezone
from importlib import import_module, metadata
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse.csgraph import connected_components

import mdpforge
from mdpforge.core.validation import validate_model
from mdpforge.models import discounted, total
from mdpforge.utils.persistence import model_cache_path

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "src/mdpforge").is_dir()
)
if not Path(mdpforge.__file__).resolve().is_relative_to(REPO_ROOT / "src"):
    raise RuntimeError("Installer mdpforge depuis ce checkout avant l'analyse.")

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 30)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})

## 1. Périmètre et protocole

`MODEL_IDS = None` sélectionne toutes les recettes des critères choisis. Pour décrire une campagne précise, renseigner ses identifiants, par exemple `{"discounted:rooms", "total:rooms"}`, et ses presets. Un identifiant est une recette du catalogue, pas une famille indépendante. Le critère moyen réutilise les modèles discounted : il ne constitue pas un troisième catalogue de problèmes.

Toutes les instances sélectionnées sont analysées sans plafond de taille ; les limites natives de construction exposées par les lecteurs sont désactivées. Le cache garde sa politique habituelle `readwrite` : ni purge ni calcul de référence. L'état des générateurs globaux NumPy et Python est restauré autour de chaque construction. La graine de cette analyse ne remplace pas les graines internes propres aux recettes.

In [ ]:
CRITERIA = ("discounted", "total")
MODEL_IDS = None
CATEGORIES = None
PRESETS = ("tiny",)
ANALYZE_MATRICES = True
SEED = 0
CACHE_MODE = "readwrite"
ANALYZE_GRAPH = True
ATOL = 1e-12
PREVIEW_ID = "discounted:rooms"
PREVIEW_PRESET = "tiny"
PREVIEW_ACTION = 0
PREVIEW_STATES = 60
EXPORT = True
OUTPUT_ROOT = REPO_ROOT / "artifacts/tmp/model_diversity"

# Annotations relues : parenté de tâche, sans prétendre à l'équivalence des MDP.
# Une clé nue s'applique aux deux critères ; une clé qualifiée est prioritaire.
ANNOTATIONS = {
    "rooms": {
        "family": "four_rooms",
        "task": "Navigation vers une sortie dans quatre pièces",
        "evidence": "Description et référence Hengst dans les deux modules rooms.py.",
    },
    "maze_prims": {
        "family": "generated_maze",
        "task": "Navigation dans un labyrinthe généré",
        "evidence": "Description du module maze_prims.py : générateur de Prim.",
    },
    "maze_wilson": {
        "family": "generated_maze",
        "task": "Navigation dans un labyrinthe généré",
        "evidence": "Description du module maze_wilson.py : générateur de Wilson.",
    },
    "maze_backtrack": {
        "family": "generated_maze",
        "task": "Navigation dans un labyrinthe généré",
        "evidence": "Description du module maze_backtrack.py : génération par retour arrière.",
    },
    "discounted:consensus": {
        "family": "shared_coin_consensus",
        "origin": "PRISM",
        "task": "Coût de terminaison du consensus",
        "evidence": "Description et docstring du module consensus.py.",
    },
    "discounted:qvbs_consensus": {
        "family": "shared_coin_consensus",
        "origin": "qvbs",
        "task": "Coût de terminaison du consensus",
        "evidence": "Description et docstring du module qvbs_consensus.py.",
    },
    "discounted:jani_rabin": {
        "family": "rabin_mutual_exclusion",
        "origin": "jani-models",
        "evidence": "Description Rabin mutual exclusion et source_collection du module.",
    },
    "discounted:prism_rabin": {
        "family": "rabin_mutual_exclusion",
        "origin": "PRISM",
        "evidence": "Description et docstring du module prism_rabin.py.",
    },
}

# Renseigner family, origin, task, objective, temporal_type, evidence si connus.
# Une valeur absente reste inconnue, même si un nom suggère une parenté.
CATALOGUES = {"discounted": discounted, "total": total}
if not CRITERIA or set(CRITERIA) - CATALOGUES.keys():
    raise ValueError("CRITERIA doit contenir discounted et/ou total.")
if not PRESETS or len(set(PRESETS)) != len(PRESETS):
    raise ValueError("PRESETS doit être une séquence non vide sans doublons.")

In [ ]:
catalogue_rows = []
preset_rows = []
entries_by_id = {}
for criterion in CRITERIA:
    for entry in CATALOGUES[criterion].list_models():
        model_id = f"{criterion}:{entry['name']}"
        entries_by_id[model_id] = entry
        annotation = ANNOTATIONS.get(model_id, ANNOTATIONS.get(entry["name"], {}))
        row = {
            "model_id": model_id,
            "criterion": criterion,
            "recipe": entry["name"],
            "category": entry.get("category"),
            "description": entry.get("description"),
            "reference": entry.get("reference"),
            "available": entry.get("available", True),
            "unavailable_reason": entry.get("reason"),
            "family": annotation.get("family"),
            "task": annotation.get("task"),
            "objective": annotation.get("objective"),
            "temporal_type": annotation.get("temporal_type"),
            "origin": annotation.get("origin", entry.get("source_collection")),
            "annotation_evidence": annotation.get("evidence"),
            "source_url": entry.get("source_url"),
        }
        row["family_reviewed"] = bool(row["family"] and row["annotation_evidence"])
        if row["family"] and not row["family_reviewed"]:
            raise ValueError(
                f"Une famille annotée doit avoir une justification : {model_id}"
            )
        row["selected"] = (MODEL_IDS is None or model_id in MODEL_IDS) and (
            CATEGORIES is None or row["category"] in CATEGORIES
        )
        catalogue_rows.append(row)
        for preset, options in entry.get("sizes", {}).items():
            preset_rows.append(
                {
                    "model_id": model_id,
                    "preset": preset,
                    "requested_states": options.get("requested_state_dim"),
                    "parameters": options.get("parameters"),
                    "preset_available": options.get("parameters") is not None,
                    "preset_reason": options.get("reason"),
                }
            )

catalogue = pd.DataFrame(catalogue_rows)
presets = pd.DataFrame(preset_rows)
if MODEL_IDS is not None and set(MODEL_IDS) - set(catalogue.model_id):
    raise ValueError(
        f"Identifiants inconnus : {set(MODEL_IDS) - set(catalogue.model_id)}"
    )
if CATEGORIES is not None and set(CATEGORIES) - set(catalogue.category):
    raise ValueError(
        f"Catégories inconnues : {set(CATEGORIES) - set(catalogue.category)}"
    )
if set(PRESETS) - set(presets.preset):
    raise ValueError(f"Presets inconnus : {set(PRESETS) - set(presets.preset)}")
selected = catalogue.loc[catalogue.selected].copy()
if selected.empty:
    raise ValueError("La sélection est vide.")
display(
    pd.DataFrame(
        {
            "catalogue": [len(catalogue), catalogue.available.sum()],
            "selection": [len(selected), selected.available.sum()],
        },
        index=["recettes", "recettes déclarées disponibles"],
    )
)
display(selected.sort_values(["category", "model_id"]))

## 2. Couverture des tâches et liens entre recettes

Les catégories sont celles du catalogue, sans reclassement automatique. Les familles sont des regroupements explicites de tâches, documentés dans `ANNOTATIONS` ; cette première liste est partielle. La même famille peut réunir plusieurs dynamiques ou objectifs. Les origines non renseignées restent inconnues : un préfixe de module n'est pas une preuve de provenance.

Le nombre exact de familles ne peut pas être établi tant que les parentés restantes n'ont pas été relues. Le tableau affiche donc les familles annotées et les recettes sans regroupement séparément. Les comptages d'instances ne sont jamais présentés comme des comptages de familles indépendantes.

In [ ]:
coverage = (
    catalogue.groupby("category")
    .size()
    .rename("catalogue")
    .to_frame()
    .join(selected.groupby("category").size().rename("selection"))
    .fillna(0)
    .astype(int)
)
display(coverage)
coverage.plot.barh(
    title="Recettes par catégorie : catalogue et sélection", xlabel="Recettes"
)
plt.tight_layout()
plt.show()

family_coverage = selected.groupby("category").agg(
    recipes=("model_id", "size"),
    annotated_families=("family", "nunique"),
    unreviewed_recipes=("family_reviewed", lambda values: int((~values).sum())),
)
display(family_coverage)
display(pd.crosstab(selected.category, selected.origin.fillna("inconnue")))
family_groups = (
    selected.loc[selected.family_reviewed]
    .groupby("family")
    .agg(
        recipes=("model_id", list),
        criteria=("criterion", lambda x: sorted(set(x))),
        evidence=("annotation_evidence", list),
    )
)
display(family_groups)
display(
    selected.loc[~selected.family_reviewed, ["model_id", "description", "reference"]]
)

preset_coverage = presets.merge(selected[["model_id", "category"]], on="model_id")
display(
    pd.crosstab(
        preset_coverage.category,
        preset_coverage.preset,
        values=preset_coverage.preset_available,
        aggfunc="sum",
    ).fillna(0)
)
display(
    preset_coverage.loc[
        ~preset_coverage.preset_available, ["model_id", "preset", "preset_reason"]
    ]
)

## 3. Définition des mesures numériques

Pour chaque instance, `P[a][s,t]` est la probabilité de passer de `s` à `t` sous l'action `a`, et `R[s,a]` la récompense moyenne immédiate. Les statistiques donnent le même poids à chaque couple état-action ; elles ne décrivent pas une distribution de visite sous une politique. Les graphiques de synthèse donnent le même poids à chaque instance analysée ; des presets multiples surreprésentent donc les recettes qui en ont davantage.

Les transitions sont étudiées sur des copies CSR canoniques : les doublons sont additionnés et les zéros stockés sont retirés, sans modifier le modèle. On mesure le nombre de probabilités strictement positives, la densité `nnz / (A × S²)`, la distribution des nombres de successeurs, la proportion de lignes déterministes, l'entropie moyenne en bits et la probabilité moyenne de rester sur place. L'entropie vaut zéro pour une transition déterministe ; elle augmente lorsque la probabilité se répartit entre plusieurs successeurs.

Un état est dit absorbant si toutes les actions restent sur cet état avec probabilité 1, à `ATOL` près. Cela ne suffit pas à lui donner le sens de « fin de tâche ». Les états terminaux déclarés par le modèle sont comptés séparément ; l'absence de déclaration donne une valeur manquante, pas zéro.

Les récompenses sont décrites par leurs quantiles, moyenne, écart-type, amplitude et proportions nulles, positives et négatives. On distingue une récompense globale constante et une récompense qui varie entre actions dans un même état. La valeur zéro et l'égalité des récompenses sont traitées exactement ; aucune normalisation n'est appliquée.

Deux actions d'un état sont identiques ici si leurs distributions de transition et leurs récompenses sont exactement égales. Cette mesure repère notamment les emplacements d'actions répétés par certains imports ; elle ne prouve pas que les actions distinctes ont des qualités différentes. Le calcul est effectué pour toutes les instances analysées.

In [ ]:
def transition_metrics(transitions):
    """Décrire les probabilités et les supports sans changer les matrices."""
    states = transitions[0].shape[0]
    absorbing = np.ones(states, dtype=bool)
    degrees = []
    action_rows = []
    for action, matrix in enumerate(transitions):
        degree = np.diff(matrix.indptr)
        diagonal = matrix.diagonal()
        row_sum = np.asarray(matrix.sum(axis=1)).ravel()
        deterministic = (degree == 1) & np.isclose(row_sum, 1, atol=ATOL, rtol=0)
        absorbing &= (degree == 1) & np.isclose(diagonal, 1, atol=ATOL, rtol=0)
        entropy = -np.sum(matrix.data * np.log2(matrix.data)) / states
        action_rows.append(
            {
                "action": action,
                "transition_nnz": matrix.nnz,
                "transition_density": matrix.nnz / states**2,
                "successors_min": degree.min(),
                "successors_median": np.median(degree),
                "successors_p90": np.quantile(degree, 0.9),
                "successors_max": degree.max(),
                "successors_mean": degree.mean(),
                "deterministic_share": deterministic.mean(),
                "self_loop_mass_mean": diagonal.mean(),
                "entropy_bits_mean": entropy,
                "row_sum_error_max": np.max(np.abs(row_sum - 1)),
            }
        )
        degrees.append(degree)
    degrees = np.concatenate(degrees)
    actions = len(transitions)
    table = pd.DataFrame(action_rows)
    result = {
        "transition_nnz": int(table.transition_nnz.sum()),
        "transition_density": table.transition_nnz.sum() / (actions * states**2),
        "successors_min": int(degrees.min()),
        "successors_median": np.median(degrees),
        "successors_p90": np.quantile(degrees, 0.9),
        "successors_max": int(degrees.max()),
        "successors_mean": degrees.mean(),
        "deterministic_share": table.deterministic_share.mean(),
        "self_loop_mass_mean": table.self_loop_mass_mean.mean(),
        "entropy_bits_mean": table.entropy_bits_mean.mean(),
        "absorbing_states": int(absorbing.sum()),
        "absorbing_share": absorbing.mean(),
        "row_sum_error_max": table.row_sum_error_max.max(),
    }
    return result, action_rows


def reward_metrics(rewards):
    """Décrire les valeurs de récompense et leur variation entre actions."""
    return {
        "reward_min": rewards.min(),
        "reward_p10": np.quantile(rewards, 0.1),
        "reward_median": np.median(rewards),
        "reward_p90": np.quantile(rewards, 0.9),
        "reward_max": rewards.max(),
        "reward_mean": rewards.mean(),
        "reward_std": rewards.std(),
        "reward_range": np.ptp(rewards),
        "reward_zero_share": np.mean(rewards == 0),
        "reward_positive_share": np.mean(rewards > 0),
        "reward_negative_share": np.mean(rewards < 0),
        "reward_constant": bool(np.all(rewards == rewards.flat[0])),
        "reward_action_varying_state_share": np.mean(np.ptp(rewards, axis=1) != 0),
    }


def action_metrics(transitions, rewards):
    """Compter les actions distinctes par état par comparaison exacte."""
    actions = len(transitions)
    counts = np.ones(rewards.shape[0], dtype=int)
    for action in range(1, actions):
        repeated = np.zeros(rewards.shape[0], dtype=bool)
        for previous in range(action):
            difference = transitions[action] - transitions[previous]
            difference.eliminate_zeros()
            repeated |= (np.diff(difference.indptr) == 0) & (
                rewards[:, action] == rewards[:, previous]
            )
        counts += ~repeated
    return {
        "action_comparison_status": "computed",
        "distinct_actions_mean": counts.mean(),
        "distinct_action_share": counts.mean() / actions,
        "single_choice_state_share": np.mean(counts == 1),
    }

Le graphe optionnel réunit les transitions de probabilité strictement positive de toutes les actions. Ses composantes fortement connexes sont des groupes d'états où chaque état peut rejoindre les autres dans ce graphe. Les composantes sans sortie et les cycles entre états distincts sont également comptés. Un parcours peut demander des actions différentes selon l'état : ces mesures ne certifient ni la terminaison sous toute politique, ni une propriété unichain, ni la finitude du rendement total. Les boucles sur soi sont décrites séparément.

In [ ]:
def graph_metrics(transitions):
    """Décrire le graphe réunissant les supports de toutes les actions."""
    if not ANALYZE_GRAPH:
        return {"graph_status": "disabled"}
    graph = transitions[0].astype(bool)
    for matrix in transitions[1:]:
        graph = graph.maximum(matrix.astype(bool))
    count, labels = connected_components(graph, directed=True, connection="strong")
    sizes = np.bincount(labels, minlength=count)
    edges = graph.tocoo()
    crossing = labels[edges.row] != labels[edges.col]
    outgoing_components = np.unique(labels[edges.row[crossing]])
    return {
        "graph_status": "computed",
        "scc_count": count,
        "largest_scc_share": sizes.max() / graph.shape[0],
        "sink_scc_count": count - len(outgoing_components),
        "nontrivial_cycle_state_share": np.mean(sizes[labels] > 1),
    }


def matrix_fingerprint(transitions, rewards):
    """Identifier les matrices canoniques dans leur ordre actuel."""
    digest = hashlib.sha256()
    for matrix in transitions:
        digest.update(np.asarray(matrix.shape, dtype="<i8").tobytes())
        for values, dtype in (
            (matrix.indptr, "<i8"),
            (matrix.indices, "<i8"),
            (matrix.data, "<f8"),
        ):
            digest.update(np.asarray(values, dtype=dtype).tobytes())
    digest.update(np.asarray(rewards.shape, dtype="<i8").tobytes())
    digest.update(np.asarray(rewards, dtype="<f8").tobytes())
    return digest.hexdigest()


def declared_count(model, plural, singular):
    """Compter une déclaration disponible sans inférer sa signification."""
    if hasattr(model, plural):
        values = getattr(model, plural)
        return len(values) if values is not None else np.nan
    if hasattr(model, singular):
        return 1 if getattr(model, singular) is not None else np.nan
    return np.nan

In [ ]:
plan = (
    selected[["model_id", "criterion", "recipe", "category", "available"]]
    .merge(pd.DataFrame({"preset": PRESETS}), how="cross")
    .merge(presets, how="left", on=["model_id", "preset"])
)
display(
    plan[
        [
            "model_id",
            "preset",
            "requested_states",
            "parameters",
            "preset_available",
            "preset_reason",
        ]
    ]
)

In [ ]:
def analyze_instance(item):
    """Construire une instance puis en extraire seulement des descriptions."""
    row = dict(item)
    row.update(status="pending", reason=None, stage="selection", warnings=None)
    if not row["available"]:
        row.update(
            status="unavailable", reason=entries_by_id[row["model_id"]].get("reason")
        )
        return row, [], None
    if pd.isna(row["preset_available"]) or not row["preset_available"]:
        reason = (
            row["preset_reason"] if pd.notna(row["preset_reason"]) else "Preset absent."
        )
        row.update(status="unavailable_preset", reason=reason)
        return row, [], None
    if not ANALYZE_MATRICES:
        row.update(status="disabled", reason="Inventaire seul.")
        return row, [], None
    numpy_state, python_state = np.random.get_state(), random.getstate()
    model = None
    action_rows = []
    preview_data = None
    start = perf_counter()
    caught = []
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            np.random.seed(SEED)
            random.seed(SEED)
            row["stage"] = "construction"
            constructor = import_module(
                f"mdpforge.models.{row['criterion']}.{row['recipe']}"
            ).Model
            options = dict(row["parameters"])
            options["state_dim"] = int(row["requested_states"])
            # Désactiver aussi la limite native des lecteurs qui l'exposent.
            if "state_limit" in inspect.signature(constructor).parameters:
                options["state_limit"] = None
            row["constructor_options"] = options
            model = constructor(**options)
            row.update(
                model_name=model.name, states=model.state_dim, actions=model.action_dim
            )
            expected_shape = (model.state_dim, model.action_dim)
            row["cache_present_before"] = model_cache_path(model).exists()
            model.create_model(cache=CACHE_MODE)
            row["build_seconds"] = perf_counter() - start
            if (model.state_dim, model.action_dim) != expected_shape:
                raise ValueError(
                    "Les dimensions après create_model diffèrent du constructeur."
                )
            row["model_config"] = model.get_config()
            row["declared_terminal_states"] = declared_count(
                model, "terminal_states", "terminal_state"
            )
            row["declared_initial_states"] = declared_count(
                model, "initial_states", "initial_state"
            )
            row["state_action_pairs"] = model.state_dim * model.action_dim
            row["state_target_ratio"] = model.state_dim / row["requested_states"]
            row["stage"] = "matrix_validation"
            validate_model(model)
            row["matrix_validation"] = "passed"
            row["stage"] = "description"
            transitions = []
            for matrix in model.transition_matrix:
                matrix = matrix.copy()
                matrix.sum_duplicates()
                matrix.eliminate_zeros()
                matrix.sort_indices()
                transitions.append(matrix)
            rewards = model.reward_matrix
            metrics, action_rows = transition_metrics(transitions)
            row.update(metrics)
            row.update(reward_metrics(rewards))
            row.update(action_metrics(transitions, rewards))
            row.update(graph_metrics(transitions))
            row["matrix_sha256"] = matrix_fingerprint(transitions, rewards)
            row["matrix_storage_mib"] = (
                sum(
                    m.data.nbytes + m.indices.nbytes + m.indptr.nbytes
                    for m in model.transition_matrix
                )
                + rewards.nbytes
            ) / 2**20
            for action_row in action_rows:
                action = action_row["action"]
                action_row.update(
                    model_id=row["model_id"],
                    preset=row["preset"],
                    reward_min=rewards[:, action].min(),
                    reward_max=rewards[:, action].max(),
                    reward_zero_share=np.mean(rewards[:, action] == 0),
                )
            if (row["model_id"], row["preset"]) == (PREVIEW_ID, PREVIEW_PRESET):
                if not 0 <= PREVIEW_ACTION < model.action_dim:
                    raise ValueError(
                        "PREVIEW_ACTION n'est pas une action de cette instance."
                    )
                stop = min(PREVIEW_STATES, model.state_dim)
                preview_data = {
                    "transition": transitions[PREVIEW_ACTION][:stop, :stop].toarray(),
                    "reward": rewards[:stop, :].copy(),
                    "successors": np.concatenate(
                        [np.diff(m.indptr) for m in transitions]
                    ),
                    "reward_values": rewards.ravel().copy(),
                    "states": model.state_dim,
                    "actions": model.action_dim,
                }
            row.update(status="analyzed", stage="complete")
    except ModuleNotFoundError as exc:
        row.update(status="missing_dependency", reason=f"{type(exc).__name__}: {exc}")
    except Exception as exc:
        row.update(status="error", reason=f"{type(exc).__name__}: {exc}")
        action_rows, preview_data = [], None
    finally:
        row["warnings"] = [str(item.message) for item in caught]
        row["elapsed_seconds"] = perf_counter() - start
        np.random.set_state(numpy_state)
        random.setstate(python_state)
        del model
        gc.collect()
    return row, action_rows, preview_data


instance_rows, per_action_rows, preview = [], [], None
for index, item in enumerate(plan.to_dict("records"), start=1):
    row, action_rows, image_data = analyze_instance(item)
    instance_rows.append(row)
    per_action_rows.extend(action_rows)
    if image_data is not None:
        preview = image_data
    print(
        f"{index}/{len(plan)} {item['model_id']} [{item['preset']}]: {row['status']}",
        flush=True,
    )

instances = pd.DataFrame(instance_rows)
per_action = pd.DataFrame(per_action_rows)
metrics = instances.loc[instances.status.eq("analyzed")].copy()
display(
    instances.groupby(["criterion", "preset", "status"])
    .size()
    .rename("instances")
    .to_frame()
)
display(
    instances.loc[
        ~instances.status.eq("analyzed"),
        ["model_id", "preset", "status", "stage", "reason"],
    ]
)
display(instances.loc[instances.warnings.map(bool), ["model_id", "preset", "warnings"]])

## 4. Couverture réellement analysée

Les graphiques suivants décrivent seulement les instances `analyzed`. Les erreurs et dépendances manquantes ne sont pas des propriétés du problème. Le contrôle des matrices vérifie les dimensions, les probabilités et les valeurs finies ; il ne certifie aucune solution. Les durées concernent le chargement, la construction et la description, avec le cache indiqué, et ne mesurent pas la difficulté de résolution.

In [ ]:
measured_coverage = instances.groupby(["category", "preset"]).agg(
    planned=("model_id", "size"),
    analyzed=("status", lambda values: int(values.eq("analyzed").sum())),
)
measured_coverage["analyzed_share"] = (
    measured_coverage.analyzed / measured_coverage.planned
)
display(measured_coverage)
if not metrics.empty:
    diagnostic_columns = [
        "action_comparison_status",
        "graph_status",
        "matrix_validation",
    ]
    for column in diagnostic_columns:
        display(
            metrics.groupby(["preset", column]).size().rename("instances").to_frame()
        )
    display(
        metrics[
            [
                "model_id",
                "preset",
                "requested_states",
                "states",
                "actions",
                "state_target_ratio",
                "matrix_storage_mib",
                "build_seconds",
                "cache_present_before",
            ]
        ]
    )

## 5. Dimensions et transitions

Les noms `tiny`, `small`, `medium`, `large` ne garantissent ni un ordre des dimensions réelles ni une difficulté commune entre recettes. Le nombre de successeurs complète la densité, qui dépend fortement du nombre d'états. Chaque point représente une instance ; les catégories fournissent une première lecture, sans présumer de son appartenance à une famille indépendante.

In [ ]:
if not metrics.empty:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for category, group in metrics.groupby("category"):
        axes[0, 0].scatter(group.states, group.actions, label=category, alpha=0.7)
        axes[0, 1].scatter(group.states, group.successors_mean, alpha=0.7)
        axes[1, 0].scatter(group.states, group.transition_density, alpha=0.7)
        axes[1, 1].scatter(
            group.deterministic_share, group.entropy_bits_mean, alpha=0.7
        )
    for axis in axes[0]:
        axis.set_xscale("log")
        axis.set_yscale("log")
    axes[1, 0].set_xscale("log")
    axes[1, 0].set_yscale("log")
    axes[0, 0].set(xlabel="États S", ylabel="Actions A", title="Dimensions réelles")
    axes[0, 1].set(xlabel="États S", ylabel="Successeurs moyens", title="Branchement")
    axes[1, 0].set(
        xlabel="États S", ylabel="nnz / (A × S²)", title="Densité des transitions"
    )
    axes[1, 1].set(
        xlabel="Proportion de lignes déterministes",
        ylabel="Entropie moyenne (bits)",
        title="Intensité de l'aléa",
    )
    axes[0, 0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    display(
        metrics[
            [
                "model_id",
                "preset",
                "transition_nnz",
                "successors_min",
                "successors_median",
                "successors_p90",
                "successors_max",
                "deterministic_share",
                "self_loop_mass_mean",
                "row_sum_error_max",
            ]
        ]
    )

## 6. Récompenses et décisions

L'amplitude des récompenses est conservée dans les unités natives. Les valeurs négatives ne sont pas automatiquement étiquetées « coûts » : le sens de l'objectif doit être lu dans la description ou annoté. Une récompense identique entre actions n'implique pas l'absence de décision, puisque les transitions peuvent différer.

In [ ]:
if not metrics.empty:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for category, group in metrics.groupby("category"):
        axes[0].scatter(
            group.reward_zero_share, group.reward_std, label=category, alpha=0.7
        )
        axes[1].scatter(group.actions, group.distinct_actions_mean, alpha=0.7)
        axes[2].scatter(group.absorbing_share, group.self_loop_mass_mean, alpha=0.7)
    axes[0].set(
        xlabel="Proportion de récompenses nulles", ylabel="Écart-type des récompenses"
    )
    axes[0].set_yscale("symlog", linthresh=1e-3)
    axes[1].set(
        xlabel="Emplacements d'actions A", ylabel="Actions distinctes moyennes / état"
    )
    axes[2].set(
        xlabel="Proportion d'états absorbants",
        ylabel="Probabilité moyenne de rester sur place",
    )
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    display(
        metrics[
            [
                "model_id",
                "preset",
                "reward_min",
                "reward_p10",
                "reward_median",
                "reward_p90",
                "reward_max",
                "reward_mean",
                "reward_std",
                "reward_zero_share",
                "reward_positive_share",
                "reward_negative_share",
                "reward_constant",
                "reward_action_varying_state_share",
                "distinct_action_share",
                "single_choice_state_share",
            ]
        ]
    )

## 7. Lecture d'une instance

L'aperçu montre seulement les premiers états, dans leur ordre natif, et une action choisie. Le bloc de transition peut omettre de la probabilité qui sort de ce bloc ; ses lignes ne sont donc pas nécessairement de somme 1. L'ordre des états peut masquer ou créer une apparence de structure. Les histogrammes, eux, portent sur tous les couples état-action de cette instance. Les indices d'actions sont des emplacements locaux, sans sens commun garanti entre états ou modèles.

In [ ]:
if preview is None:
    print(
        f"Pas d'aperçu pour {PREVIEW_ID} [{PREVIEW_PRESET}] : voir le bilan d'analyse."
    )
else:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    image = axes[0, 0].imshow(
        preview["transition"], aspect="auto", interpolation="nearest", vmin=0, vmax=1
    )
    axes[0, 0].set(
        title=f"P[{PREVIEW_ACTION}] : bloc des premiers états",
        xlabel="État suivant",
        ylabel="État courant",
    )
    fig.colorbar(image, ax=axes[0, 0], label="Probabilité")
    image = axes[0, 1].imshow(
        preview["reward"], aspect="auto", interpolation="nearest", cmap="coolwarm"
    )
    axes[0, 1].set(
        title="R : premiers états, toutes les actions", xlabel="Action", ylabel="État"
    )
    fig.colorbar(image, ax=axes[0, 1], label="Récompense native")
    successors, counts = np.unique(preview["successors"], return_counts=True)
    axes[1, 0].bar(successors, counts)
    axes[1, 0].set(xlabel="Nombre de successeurs", ylabel="Couples état-action")
    axes[1, 1].hist(preview["reward_values"], bins=40)
    axes[1, 1].set(xlabel="Récompense", ylabel="Couples état-action")
    fig.suptitle(
        f"{PREVIEW_ID} [{PREVIEW_PRESET}] — S={preview['states']}, A={preview['actions']}"
    )
    plt.tight_layout()
    plt.show()
    display(
        per_action.loc[
            (per_action.model_id == PREVIEW_ID) & (per_action.preset == PREVIEW_PRESET)
        ]
    )

## 8. Couverture structurelle, proximités et limites

Les classes ci-dessous servent à lire la couverture : elles ne constituent pas une mesure universelle de diversité. « Mixte » signifie que certaines lignes sont déterministes et d'autres stochastiques. Un état absorbant observé n'établit pas qu'il sera atteint. Les proportions de récompenses sont conservées dans les tables détaillées.

Les empreintes signalent des matrices identiques après canonicalisation, dans le même ordre d'états et d'actions. Elles peuvent révéler deux presets donnant la même instance. Elles ne détectent pas les équivalences après permutation, les parentés de tâche ou toutes les transformations possibles des récompenses. Aucun score global ni classement de difficulté n'est calculé.

In [ ]:
if not metrics.empty:
    metrics["dynamics_class"] = np.select(
        [metrics.deterministic_share == 1, metrics.deterministic_share == 0],
        ["déterministe", "stochastique partout"],
        default="mixte",
    )
    metrics["absorption_class"] = np.where(
        metrics.absorbing_states > 0, "avec état absorbant", "sans état absorbant"
    )
    metrics["reward_class"] = np.select(
        [metrics.reward_constant.to_numpy(dtype=bool), metrics.reward_zero_share > 0],
        ["constante", "variable avec zéros"],
        default="variable sans zéro",
    )
    for property_name in ("dynamics_class", "absorption_class", "reward_class"):
        print(f"Instances par catégorie et {property_name}")
        display(pd.crosstab(metrics.category, metrics[property_name]))
    structural_coverage = pd.crosstab(
        [metrics.category, metrics.dynamics_class],
        [metrics.absorption_class, metrics.reward_class],
    )
    display(structural_coverage)
    graph_columns = [
        "model_id",
        "preset",
        "absorbing_states",
        "declared_terminal_states",
        "declared_initial_states",
        "graph_status",
        "scc_count",
        "largest_scc_share",
        "sink_scc_count",
        "nontrivial_cycle_state_share",
    ]
    display(metrics.reindex(columns=graph_columns))
    duplicates = metrics.loc[
        metrics.duplicated("matrix_sha256", keep=False)
    ].sort_values("matrix_sha256")
    display(duplicates[["model_id", "preset", "states", "actions", "matrix_sha256"]])
    display(
        metrics.groupby(["category", "preset"])[
            [
                "states",
                "actions",
                "successors_mean",
                "deterministic_share",
                "reward_zero_share",
            ]
        ].agg(["min", "median", "max"])
    )

annotation_audit = selected.assign(
    missing_description=selected.description.fillna("").eq(""),
    missing_family=~selected.family_reviewed,
    missing_origin=selected.origin.isna(),
    missing_objective=selected.objective.isna(),
    missing_temporal_type=selected.temporal_type.isna(),
)
display(
    annotation_audit[
        [
            "model_id",
            "missing_description",
            "missing_family",
            "missing_origin",
            "missing_objective",
            "missing_temporal_type",
        ]
    ]
)

### Questions à examiner après chaque relance

- Quelles catégories et familles dominent la sélection ? La réponse change-t-elle en comptant les instances ?
- Les domaines représentés couvrent-ils plusieurs formes de dynamique, de décision et de récompense ?
- Les nouvelles recettes apportent-elles une tâche différente, une structure différente, ou surtout une nouvelle taille ?
- Quels modèles restent sans annotation ou indisponibles ? Ces exclusions concentrent-elles l'analyse sur certains domaines ?
- Plusieurs presets donnent-ils exactement les mêmes matrices ? Les structures changent-elles avec la taille lorsque plusieurs presets ont été analysés ?

Une case vide est une absence observée dans le périmètre analysé, pas la preuve d'un manque scientifique. La portée dépend des catégories du catalogue, de la qualité des annotations et des presets construits. Une seule graine et un seul preset ne décrivent pas toute la variété d'un générateur. La connectivité, les récompenses et les dimensions seules ne permettent pas de prédire les performances des solveurs.

## 9. Export et comparaison entre relances

Chaque relance exportée crée un répertoire daté sous `artifacts/tmp/model_diversity/`, avec inventaire, presets, résultats, descriptions par action, annotations et manifeste. Les tableaux complets peuvent être filtrés dans le notebook ou lus depuis les CSV. Ce dossier est ignoré par Git ; copier une campagne à conserver vers un emplacement de référence choisi explicitement.

La comparaison utilise le dernier export ayant exactement le même protocole et le même périmètre de filtres. Elle distingue les recettes ajoutées ou retirées, les métadonnées modifiées, les changements de statut et les changements de matrices parmi les instances analysées dans les deux relances. Une empreinte différente peut aussi venir d'une graine, d'une dépendance ou d'un ordre d'états différent. La présence préalable du cache ne prouve pas qu'il a été utilisé ; une modification de recette conservant un ancien nom peut nécessiter un rafraîchissement explicite du cache.

In [ ]:
def package_version(name):
    """Lire une version installée si elle est disponible."""
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def git_output(*arguments):
    """Lire l'état du dépôt sans le modifier."""
    try:
        return subprocess.run(
            ["git", "-C", str(REPO_ROOT), *arguments],
            check=True,
            capture_output=True,
            text=True,
            timeout=5,
        ).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        return None


protocol = {
    "schema": 1,
    "criteria": list(CRITERIA),
    "model_ids": None if MODEL_IDS is None else sorted(MODEL_IDS),
    "categories": None if CATEGORIES is None else sorted(CATEGORIES),
    "presets": list(PRESETS),
    "analyze_matrices": ANALYZE_MATRICES,
    "seed": SEED,
    "cache_mode": CACHE_MODE,
    "atol": ATOL,
    "analyze_graph": ANALYZE_GRAPH,
    "resource_limits": None,
}
notebook_path = REPO_ROOT / "notebooks/model_diversity.ipynb"
manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "protocol": protocol,
    "git_commit": git_output("rev-parse", "HEAD"),
    "git_status": git_output("status", "--porcelain"),
    "notebook_sha256": hashlib.sha256(notebook_path.read_bytes()).hexdigest(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {
        name: package_version(name)
        for name in ("mdpforge", "numpy", "scipy", "pandas", "matplotlib", "mazelib")
    },
    "catalogue_recipes": len(catalogue),
    "selected_recipes": len(selected),
    "planned_instances": len(instances),
    "analyzed_instances": len(metrics),
    "annotations": ANNOTATIONS,
    "solver_runs": 0,
}

previous_dir = None
if OUTPUT_ROOT.exists():
    for candidate in sorted(OUTPUT_ROOT.iterdir(), reverse=True):
        manifest_path = candidate / "manifest.json"
        if manifest_path.is_file():
            try:
                previous_manifest = json.loads(
                    manifest_path.read_text(encoding="utf-8")
                )
                if (
                    previous_manifest.get("protocol") == protocol
                    and (candidate / "inventory.json").is_file()
                ):
                    previous_dir = candidate
                    break
            except (OSError, ValueError) as exc:
                print(f"Export précédent illisible {candidate.name}: {exc}")

if previous_dir is None:
    print("Aucun export précédent avec le même protocole.")
else:
    previous_inventory = json.loads(
        (previous_dir / "inventory.json").read_text(encoding="utf-8")
    )
    current_inventory = {
        row["model_id"]: entries_by_id[row["model_id"]]
        for row in selected.to_dict("records")
    }
    added = sorted(current_inventory.keys() - previous_inventory.keys())
    removed = sorted(previous_inventory.keys() - current_inventory.keys())
    changed = sorted(
        key
        for key in current_inventory.keys() & previous_inventory.keys()
        if current_inventory[key] != previous_inventory[key]
    )
    print(f"Comparaison à {previous_dir.name}")
    display(
        pd.DataFrame(
            [
                {"change": label, "model_ids": values}
                for label, values in (
                    ("ajouts", added),
                    ("retraits", removed),
                    ("métadonnées", changed),
                )
            ]
        )
    )
    previous_instances = pd.read_csv(previous_dir / "instances.csv")
    keys = ["model_id", "preset"]
    comparison_columns = keys + [
        "status",
        "matrix_sha256",
        "states",
        "actions",
        "transition_nnz",
        "reward_min",
        "reward_max",
    ]
    comparison = previous_instances.reindex(columns=comparison_columns).merge(
        instances.reindex(columns=comparison_columns),
        on=keys,
        how="outer",
        suffixes=("_before", "_now"),
        indicator=True,
    )
    comparison["matrix_changed"] = (
        comparison.status_before.eq("analyzed")
        & comparison.status_now.eq("analyzed")
        & comparison.matrix_sha256_before.ne(comparison.matrix_sha256_now)
    )
    display(
        comparison.loc[
            comparison._merge.ne("both")
            | comparison.status_before.ne(comparison.status_now)
            | comparison.matrix_changed
        ]
    )

if EXPORT:
    output_dir = OUTPUT_ROOT / datetime.now(timezone.utc).strftime(
        "%Y-%m-%d_%H-%M-%S_%f"
    )
    output_dir.mkdir(parents=True, exist_ok=False)
    for name, table in {
        "catalogue": catalogue,
        "presets": presets,
        "instances": instances,
        "metrics": metrics,
        "per_action": per_action,
        "annotation_audit": annotation_audit,
    }.items():
        csv_table = table.copy()
        for column in csv_table:
            if csv_table[column].dtype != object:
                continue
            csv_table[column] = csv_table[column].map(
                lambda value: (
                    json.dumps(value, ensure_ascii=False, sort_keys=True)
                    if isinstance(value, (dict, list, tuple))
                    else value
                )
            )
        csv_table.to_csv(output_dir / f"{name}.csv", index=False, encoding="utf-8")
    inventory = {
        row["model_id"]: entries_by_id[row["model_id"]]
        for row in selected.to_dict("records")
    }
    (output_dir / "inventory.json").write_text(
        json.dumps(inventory, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    (output_dir / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print(f"Export : {output_dir}")
else:
    print("Export désactivé.")